In [ ]:
!pip install scikit-learn pandas matplotlib seaborn joblib

from google.colab import drive
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Mount Google Drive
drive.mount('/content/drive')

In [ ]:
# Load the generated CSV from the new separate folder
csv_file_path = "/content/drive/MyDrive/ML-CEP/csv_data/hand_landmarks_dataset.csv"
df = pd.read_csv(csv_file_path)

print(f"Dataset Shape: {df.shape}")
display(df.head())

In [ ]:
# Separate Features (X) and Labels (y)
X = df.drop(['label', 'person'], axis=1)
y = df['label']

# Split into Training and Testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples: {X_test.shape[0]}")

In [ ]:
# Define the Random Forest model
rf = RandomForestClassifier(random_state=42)

# Define Hyperparameters to tune
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# Setup Grid Search with 3-fold Cross Validation
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, 
                           cv=3, n_jobs=-1, verbose=2, scoring='accuracy')

print("Starting Hyperparameter Tuning. This may take a few minutes...")
grid_search.fit(X_train, y_train)

print(f"\nBest Parameters found: {grid_search.best_params_}")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_ * 100:.2f}%")

# Extract the best model
best_rf = grid_search.best_estimator_

In [ ]:
# Setup directory for model outputs
model_out_dir = "/content/drive/MyDrive/ML-CEP/model_output"
os.makedirs(model_out_dir, exist_ok=True)

# Save the trained model to the separate folder
model_path = os.path.join(model_out_dir, "random_forest_best_model.pkl")
joblib.dump(best_rf, model_path)
print(f"Model successfully saved to: {model_path}")

In [ ]:
# Predictions on Test Set
y_pred = best_rf.predict(X_test)

# Calculate Accuracy
test_acc = accuracy_score(y_test, y_pred)
print(f"\nFinal Test Set Accuracy: {test_acc * 100:.2f}%\n")

print("Classification Report:")
report = classification_report(y_test, y_pred)
print(report)

# Optionally save the report to text
with open(os.path.join(model_out_dir, "classification_report.txt"), "w") as f:
    f.write(report)

# Generate Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

# Plot Confusion Matrix and Save it
plt.figure(figsize=(16, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=np.unique(y), yticklabels=np.unique(y))
plt.title('Confusion Matrix - Hand Sign Detection')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig(os.path.join(model_out_dir, "confusion_matrix.png"))
plt.show()